# Scenario: Calculating the "Recall" of a Cancer Screening AI

In [11]:
import pandas as pd
import sqlite3
# AI Predictions (Positive = Thinks it's cancer, Negative = Thinks it's healthy)
ai_findings = {
    "scan_id": [301, 302, 303, 304, 305, 306],
    "patient_id": ["P-99", "P-98", "P-97", "P-96", "P-95", "P-94"],
    "ai_result": ["Positive", "Negative", "Positive", "Negative", "Positive", "Negative"]   
}

# Gold Standard Results (Actual Biopsy Results)
actual_biopsy = {
    "patient_id": ["P-99", "P-98", "P-97", "P-96", "P-95", "P-94"],
    "actual_condition": ["Positive", "Positive", "Positive", "Negative", "Negative", "Negative"]
    }
# adding the data to DataFrame
df_ai_findings = pd.DataFrame(ai_findings)
df_actual_biopsy = pd.DataFrame(actual_biopsy)
# connecting sqlite3 for temporary memory to save the dataset
connt = sqlite3.connect(":memory:")
# saving dataset to sql
df_ai_findings.to_sql("ai_results",connt, index = False, if_exists = "replace")
df_actual_biopsy.to_sql("actual_results", connt, index = False, if_exists = "replace")
# function to run the query
def run_query(query):
    return pd.read_sql(query, connt)
print("************************* Sensitivity Audit Database is ready! ***********************")


************************* Sensitivity Audit Database is ready! ***********************


# Identify the "Dangerous Misses" (False Negatives)

In [15]:
# # query for all dataset for a quick view
all_data_1 = "SELECT * FROM ai_results"
print("*************************************** all data to quick review for the first table ***************************")
display(run_query(all_data_1))
print()
all_data_2 = "SELECT * FROM actual_results"
print("*************************************** all data to quick review for the second table ***************************")
display(run_query(all_data_2))
print("///////////////////////////////////////////////////////////////////////////////////////////////////////////")
# query that joins the two tables on patient_id. Find all rows where the actual_condition was 'Positive' but the ai_result was 'Negative'
false_negatives = """
SELECT * FROM ai_results
INNER JOIN actual_results ON ai_results.patient_id =  actual_results.patient_id
WHERE actual_condition = "Positive" AND ai_result = "Negative"
"""
print("****************************** False Negative ***********************")
display(run_query(false_negatives))

*************************************** all data to quick review for the first table ***************************


,scan_id,patient_id,ai_result
0,301,P-99,Positive
1,302,P-98,Negative
2,303,P-97,Positive
3,304,P-96,Negative
4,305,P-95,Positive
5,306,P-94,Negative



*************************************** all data to quick review for the second table ***************************


,patient_id,actual_condition
0,P-99,Positive
1,P-98,Positive
2,P-97,Positive
3,P-96,Negative
4,P-95,Negative
5,P-94,Negative


///////////////////////////////////////////////////////////////////////////////////////////////////////////
****************************** False Negative ***********************


,scan_id,patient_id,ai_result,patient_id,actual_condition
0,302,P-98,Negative,P-98,Positive


# Calculating the "True Positive" Count

In [18]:
# query that counts how many times the AI correctly identified a 'Positive' case.
true_postitives = """
SELECT COUNT(*) AS true_positives FROM ai_results
INNER JOIN actual_results ON ai_results.patient_id = actual_results.patient_id
WHERE ai_result = "Positive" AND actual_condition = "Positive"
"""
print("********************************* true_positves **********************")
display(run_query(true_postitives))


********************************* true_positves **********************


,true_positives
0,2
